In [ ]:
%matplotlib tk 
# or %matplotlib widget for interactive inline plot
import os
import hashlib
import numpy as np
import time
import json
from pathlib import Path
import h5py
import re
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button, TextBox, CheckButtons
from scipy.signal import find_peaks
# from scipy.optimize import curve_fit
from sklearn.linear_model import RANSACRegressor

# Functions

### Finding Data

In [2]:
CACHE_DIR = "drive_index_cache_files"
if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

def get_all_files(root_path, force_update=False):
    """
    Returns a list of all file paths for the given root_path.
    Creates a specific, hashed cache file for this path in the background.
    """
    # 1. Generate a unique cache filename for THIS path
    # We hash the path string to create a safe filename (e.g. "cache_5d41402abc.json")
    path_str = str(root_path)
    path_hash = hashlib.md5(path_str.encode('utf-8')).hexdigest()
    cache_file = os.path.join(CACHE_DIR, f"cache_{path_hash}.json")

    # 2. Check if cache exists and we are not forcing an update
    if os.path.exists(cache_file) and not force_update:
        print(f"Loading file list from local cache: {cache_file}...")
        t0 = time.time()
        try:
            with open(cache_file, 'r') as f:
                file_list = json.load(f)
            print(f"Loaded {len(file_list)} files in {time.time()-t0:.2f}s")
            return file_list
        except Exception as e:
            print(f"Cache corrupted ({e}), rescanning...")

    # 3. If no cache or force_update, scan the drive
    print(f"Scanning drive (this may take a while)... {root_path}")
    t0 = time.time()
    
    root = Path(root_path)
    if not root.exists():
        print(f"Warning: Path not found: {root_path}")
        return []

    # Recursively find all files
    files = [p.as_posix() for p in root.rglob('*') if p.is_file()]
    
    print(f"Scan complete. Found {len(files)} files in {time.time()-t0:.2f}s")

    # 4. Save to the specific cache file
    print(f"Saving cache to {cache_file}...")
    with open(cache_file, 'w') as f:
        json.dump(files, f)
        
    return files

def find_file_cached(root_path, substring, force_refresh=False):
    # Get the list (manages cache automatically based on root_path)
    all_files = get_all_files(root_path, force_update=force_refresh)
    
    # Perform the search in memory
    matches = [f for f in all_files if substring in f.split('/')[-1]]
    
    return matches

### Loading Data

In [3]:
def load_data(short=None, folder=None, name=None, path=None, manual_indices=None, sort=False, exclude=None):
    if (short is None or folder is None) and (name is None or path is None):
        raise ValueError("Either ('short' and 'folder') or ('name' and 'path') must be provided.")
    if short:
        filepaths = find_file_cached("Z:/POBox/Jonas Gerber/05 - Measurements (Data)/"+folder+"/", short, force_refresh=False)
        if not filepaths:
            raise FileNotFoundError(f"No files found for short='{short}' in folder='{folder}'")
        print(f"Found files: {[os.path.basename(f) for f in filepaths]}")
        # Use first match with ".hdf5" in the name. If exclude!=None, exclude files containing that substring.
        if exclude is not None:
            filepaths = [f for f in filepaths if exclude not in os.path.basename(f)]
            if not filepaths:
                raise FileNotFoundError(f"No files found for short='{short}' in folder='{folder}' after excluding '{exclude}'")
            print(f"After excluding '{exclude}', found files: {[os.path.basename(f) for f in filepaths]}")
        filepath = next((f for f in filepaths if f.endswith(".hdf5")), None)
        if filepath is None:
            raise FileNotFoundError(f"No .hdf5 files found for short='{short}' in folder='{folder}'")
        print(f"Chose file: {filepath}")
    else:
        name_extension = name+".hdf5"
        if not path.startswith("Z:/"):
            path = "Z:/POBox/Jonas Gerber/05 - Measurements (Data)"+path
        filepath = os.path.join(path, name_extension)

    data = h5py.File(filepath, 'r')
    # Extract raw data to inspect
    raw_names = data['Data']['Channel names'][:]
    
    # Clean the names: 
    # 1. Select the first element of the tuple (the name)
    # 2. Decode bytes to string
    # 3. Strip whitespace
    channel_names = [x[0].decode('utf-8').strip() for x in raw_names]
    
    print("Found channels:", channel_names)
    
    
    # 2. Define Flexible Patterns (Regex)
    # -----------------------------------
    # These patterns define what you are looking for.
    # .  = any character
    # * = zero or more times
    # |  = OR logic
    # ?  = optional character (good for _ vs space)
    patterns = {
        'V_PG':  r'PG.?Virt(?:ual)?|V.?[PC]G',      # Matches "PG Virtual" OR "V_PG" OR "V_CG"
        'B':     r'Magnet|\bB\b',                   # Matches anything with "Magnet" in it
        'I_ACx': r'I.?(?:SD)?.*AC.*x',              # Matches "I", (space/_), "SD", then "AC", then "x" (or "X")
        'I_SD':  r'I.?SD(?!.*AC)',                  # Matches "I_SD C18", "I SD C18" [potentially also finds AC!]
    }
    
    # 3. Helper Function to Get Index
    # -------------------------------
    def get_index(pattern, name_list):
        for i, name in enumerate(name_list):
            if re.search(pattern, name, re.IGNORECASE):
                return i
        if pattern not in ["I.?(SD)?.*AC.*(x|X)"]:
            raise ValueError(f"Could not find channel matching pattern: {pattern}")
        else:
            return -1

    # 4. Map Patterns to Indices
    # --------------------------
    indices = {key: get_index(pat, channel_names) for key, pat in patterns.items()}

    print("Mapped Indices:", indices)
    if manual_indices is not None:
        if len(manual_indices) == len(patterns.keys()):
            indices = {key: idx for key, idx in zip(patterns.keys(), manual_indices)}
            print("Overwrote Indices:", indices)
        else:
            print(f"Could not overwrite: manual_indices has length {len(manual_indices)}, needs length {len(patterns.keys())}")
    
    
    # 5. Extract Data Using Dynamic Indices
    # -------------------------------------
    # Now use the dictionary indices instead of hardcoded numbers
    dataset = data['Data']['Data']
    
    V_PG  = dataset[:, indices['V_PG'], :]
    if indices['I_ACx'] == -1:
        I_SD = dataset[:, indices['I_SD'], :]
        dx = V_PG[1, 0]-V_PG[0, 0]
        if dx == 0:
            dx = V_PG[0, 1]-V_PG[0, 0]
        print("dx=", dx)
        I_ACx = np.gradient(I_SD, dx, axis=1) * 1e-8
        print("Manually computed I_AC with np.gradient(I_SD,...).")
    else:
        I_ACx = dataset[:, indices['I_ACx'], :] * 1e-8
    if indices['B'] != -1:
        B = dataset[:, indices['B'], :]
        
        # n_B_values = len(np.unique(B))
        # print(f"Detected {n_B_values} unique B-field values: {np.unique(B)}")
        # # We need to only take every n_B_values-th slice to get a single B-field
        # V_PG  = V_PG[:, ::n_B_values]
        # I_ACx = I_ACx[:, ::n_B_values]

    # Convert raw data to physical units
    z_all = 1e12 * I_ACx[:, :] # 0:101]  # pA
    # z_all = np.fliplr(z_all)  # to test weak hole

    is_pg_horizontal = abs(V_PG[0, 1] - V_PG[0, 0]) > 1e-9

    if is_pg_horizontal:
        # CASE 1: Standard Orientation (PG is fast axis/columns)
        print("Detected orientation: PG varies along columns.")
        x_centers = V_PG[0, :]
        y_centers = B[:, 0]
        z_final = z_all  # No transpose needed
        
    else:
        # CASE 2: Transposed Orientation (PG is slow axis/rows)
        print("Detected orientation: PG varies along rows. Transposing data...")
        x_centers = V_PG[:, 0]   # Take the column (vertical variation)
        y_centers = B[0, :]   # Take the row (horizontal variation)
        
        # CRITICAL: Transpose Z so the plotting code (pcolormesh) 
        # still sees X as horizontal and Y as vertical.
        z_final = z_all.T

    # ---------------------------------------------------------
    # 7. SORTING (Fixes pcolormesh warning)
    # ---------------------------------------------------------
    if sort:
        # Ensure x_centers is strictly increasing
        if np.any(np.diff(x_centers) <= 0):
            sort_x = np.argsort(x_centers)
            x_centers = x_centers[sort_x]
            z_final = z_final[:, sort_x]  # Reorder columns of data

        # Ensure y_centers is strictly increasing
        if np.any(np.diff(y_centers) <= 0):
            sort_y = np.argsort(y_centers)
            y_centers = y_centers[sort_y]
            z_final = z_final[sort_y, :]  # Reorder rows of data

    if not np.isfinite(z_all).all():
        print(f"Warning: Found {np.isnan(z_all).sum()} NaNs and {np.isinf(z_all).sum()} Infs in Z-data.")
        print("-> Replacing with zeros/finite values to prevent crash.")
        z_all = np.nan_to_num(z_all, nan=0.0, posinf=np.nanmax(z_all[np.isfinite(z_all)]), neginf=0.0)

    return name, x_centers, y_centers, z_final

# Choosing data

In [19]:
search_dir = "Z:/POBox/Jonas Gerber/05 - Measurements (Data)/Stack18/"
name_substring = "D046"

# files = list(Path(search_dir).rglob(f"*{name_substring}*.hdf5"))
files = find_file_cached(search_dir, name_substring, force_refresh=False)

if files:
    for file in files:
        print(file)
else:
    print("No files found.")

Loading file list from local cache: drive_index_cache_files\cache_907577e4e3a4e61c42a26b20bb10dc08.json...
Loaded 6328 files in 0.00s
Z:/POBox/Jonas Gerber/05 - Measurements (Data)/Stack18/Titan_2025/2025/10/Data_1026/D046 QD3 BG=-9V --- PG vs Field vs bias.hdf5


In [20]:
name, x_centers, y_centers, z_all = load_data(name='B0424 6VBG 1h PG vs fieild fixed bias -1.9', \
                                              path='/EE01/Fritz2025/2025/04/Data_0417/', sort=False)

Found channels: ['PG-virtual', 'Magnet - B', 'V_PG Left C24', 'I_SD Left C18', 'I SD C18 AC x', 'I SD C18 AC theta']
Mapped Indices: {'V_PG': 0, 'B': 1, 'I_ACx': 4, 'I_SD': 3}
Detected orientation: PG varies along rows. Transposing data...


In [21]:
name, x_centers, y_centers, z_all = load_data(short='B0453', folder='EE01')

Loading file list from local cache: drive_index_cache_files\cache_751a2bd35bf7a7f27a5c4071ba1c4b51.json...
Loaded 3130 files in 0.00s
Found files: ['B0453  6VBG 1h PG vs field.hdf5']
Chose file: Z:/POBox/Jonas Gerber/05 - Measurements (Data)/EE01/Fritz2025/2025/04/Data_0429/B0453  6VBG 1h PG vs field.hdf5
Found channels: ['PG-virtual', 'Magnet - B', 'V_PG Left C24', 'I_SD Left C18', 'I SD C18 AC x', 'I SD C18 AC theta']
Mapped Indices: {'V_PG': 0, 'B': 1, 'I_ACx': 4, 'I_SD': 3}
Detected orientation: PG varies along rows. Transposing data...


In [22]:
name, x_centers, y_centers, z_all = load_data(short='B0590', folder='EE01')

Loading file list from local cache: drive_index_cache_files\cache_751a2bd35bf7a7f27a5c4071ba1c4b51.json...
Loaded 3130 files in 0.00s
Found files: ['B0590 6VBG 1e PG vs field at -400µV bias.hdf5', 'B0590.csv']
Chose file: Z:/POBox/Jonas Gerber/05 - Measurements (Data)/EE01/Fritz2025/2025/05/Data_0514/B0590 6VBG 1e PG vs field at -400µV bias.hdf5
Found channels: ['PG-virtual', 'Magnet - B', 'V_PG Left C24', 'I_SD Left C18', 'I SD C18 AC x', 'I SD C18 AC theta']
Mapped Indices: {'V_PG': 0, 'B': 1, 'I_ACx': 4, 'I_SD': 3}
Detected orientation: PG varies along rows. Transposing data...


In [27]:
name, x_centers, y_centers, z_all = load_data(name='C0057 4VBGm 1e PG BField MGs', \
                                              path='/EE01/Fritz2025 Cooldown C/2025/07/Data_0715/', sort=False)

Found channels: ['PG Virtual', 'Magnet - B', 'V_SD Left C18 C6', 'V_MG C17', 'V_PG Left C24', 'I_SD Left C18', 'I SD C18 AC x', 'I SD C18 AC theta']
Mapped Indices: {'V_PG': 0, 'B': 1, 'I_ACx': 6, 'I_SD': 5}
Detected orientation: PG varies along rows. Transposing data...


In [29]:
name, x_centers, y_centers, z_all = load_data(short='C0058', folder='EE01', sort=False)

Loading file list from local cache: drive_index_cache_files\cache_751a2bd35bf7a7f27a5c4071ba1c4b51.json...
Loaded 3130 files in 0.00s
Found files: ['C0058_1.csv', 'C0058 4VBGm 1e PG BField MGs.hdf5']
Chose file: Z:/POBox/Jonas Gerber/05 - Measurements (Data)/EE01/Fritz2025 Cooldown C/2025/07/Data_0715/C0058 4VBGm 1e PG BField MGs.hdf5
Found channels: ['PG Virtual', 'Magnet - B', 'V_SD Left C18 C6', 'V_MG C17', 'V_PG Left C24', 'I_SD Left C18', 'I SD C18 AC x', 'I SD C18 AC theta']
Mapped Indices: {'V_PG': 0, 'B': 1, 'I_ACx': 6, 'I_SD': 5}
Detected orientation: PG varies along rows. Transposing data...
-> Replacing with zeros/finite values to prevent crash.


In [25]:
name, x_centers, y_centers, z_all = load_data(short='C0196', folder='EE01')

Loading file list from local cache: drive_index_cache_files\cache_751a2bd35bf7a7f27a5c4071ba1c4b51.json...
Loaded 3130 files in 0.00s
Found files: ['C0196_8VBG_1e PG BField.hdf5', 'C0196_8VBG_1e PG BField.png']
Chose file: Z:/POBox/Jonas Gerber/05 - Measurements (Data)/EE01/Fritz2025 Cooldown C/2025/07/Data_0725/C0196_8VBG_1e PG BField.hdf5
Found channels: ['PG virtual', 'B-Field - B', 'V_PG Left C24', 'I SD AC C18 AC x', 'I_SD Left C18']
Mapped Indices: {'V_PG': 0, 'B': 1, 'I_ACx': 3, 'I_SD': 4}
Detected orientation: PG varies along rows. Transposing data...


In [ ]:
name, x_centers, y_centers, z_all = load_data(short='B149', folder='EE04', exclude='Diamond')
# no PG?

In [ ]:
name, x_centers, y_centers, z_all = load_data(short='D046', folder='Stack18')

Loading file list from local cache: drive_index_cache_files\cache_907577e4e3a4e61c42a26b20bb10dc08.json...
Loaded 6328 files in 0.03s
Found files: ['D046 QD3 BG=-9V --- PG vs Field vs bias.hdf5']
Chose file: Z:/POBox/Jonas Gerber/05 - Measurements (Data)/Stack18/Titan_2025/2025/10/Data_1026/D046 QD3 BG=-9V --- PG vs Field vs bias.hdf5
Found channels: ['PG_virt', 'Oxford IPS120 - B', 'V_SD_DC (C5-22)', 'PG (C1)', 'I_SD ACx (C5-22)', 'V_SD_ACx (C4-20)', 'I_SD DC (C5-22)', 'V_SD_DC (C4-20)', 'Theta (C5-22)', 'Theta V (C4-20)']
Mapped Indices: {'V_PG': 0, 'B': 1, 'I_ACx': 4, 'I_SD': 6}
Detected orientation: PG varies along rows. Transposing data...


In [31]:
print("x_centers shape:", x_centers.shape
      , "y_centers shape:", y_centers.shape
      , "z_all shape:", z_all.shape)

x_centers shape: (126,) y_centers shape: (1909,) z_all shape: (1909, 126)


In [ ]:
for y in y_centers:
    print(y, " ")

0.0  
0.0025  
0.005  
0.0075  
0.01  
0.0125  
0.015  
0.0175  
0.02  
0.0225  
0.025  
0.0275  
0.03  
0.0325  
0.035  
0.0375  
0.04  
0.0425  
0.045  
0.0475  
0.05  
0.0525  
0.055  
0.0575  
0.06  
0.0625  
0.065  
0.0675  
0.07  
0.0725  
0.075  
0.0775  
0.08  
0.0825  
0.085  
0.08750000000000001  
0.09  
0.0925  
0.095  
0.0975  
0.1  
0.10250000000000001  
0.105  
0.1075  
0.11  
0.1125  
0.115  
0.11750000000000001  
0.12  
0.1225  
0.125  
0.1275  
0.13  
0.1325  
0.135  
0.1375  
0.14  
0.14250000000000002  
0.145  
0.1475  
0.15  
0.1525  
0.155  
0.1575  
0.16  
0.1625  
0.165  
0.1675  
0.17  
0.17250000000000001  
0.17500000000000002  
0.1775  
0.18  
0.1825  
0.185  
0.1875  
0.19  
0.1925  
0.195  
0.1975  
0.2  
0.2025  
0.20500000000000002  
0.20750000000000002  
0.21  
0.2125  
0.215  
0.2175  
0.22  
0.2225  
0.225  
0.2275  
0.23  
0.2325  
0.23500000000000001  
0.23750000000000002  
0.24  
0.2425  
0.245  
0.2475  
0.25  
0.2525  
0.255  
0.2575  
0.26  
0.262

In [ ]:
# 1. Find turning points (Peaks and Valleys)
# We look for peaks in the data (+) and peaks in the inverted data (-)
pks_max, _ = find_peaks(y_centers, prominence=0.01) # Find tops (0.3)
pks_min, _ = find_peaks(-y_centers, prominence=0.01) # Find bottoms (0.0)

# Combine all turning points
turning_points = np.concatenate((pks_max, pks_min))
turning_points.sort()

# 2. Determine Start Index
if len(turning_points) > 0:
    # The last sweep starts at the very last turning point
    start_idx = turning_points[-1]
    print(f"Found turning point at index {start_idx}. Slicing last sweep.")
else:
    # No turning points found? It's already a single monotonic line.
    start_idx = 0
    print("No turning points found. Using full data.")

# 3. Slice the data
y_centers = y_centers[start_idx:]
z_all = z_all[start_idx:, :]

# 4. Final Monotonicity Check (Reverse if needed)
# If the sweep goes High -> Low (e.g. 0.3 -> 0.0), we reverse it 
# so the plot axes work normally (Low -> High).
if y_centers[0] > y_centers[-1]:
    print("Sweep is High->Low. Reversing to fit plot axes.")
    y_centers = y_centers[::-1]
    z_all = z_all[::-1, :]
else:
    print("Sweep is Low->High. Keeping as is.")

print(f"Final Y shape: {y_centers.shape}")

# np.unique returns the sorted unique elements and their original indices
y_centers, unique_indices = np.unique(y_centers, return_index=True)

# Apply the exact same filtering/reordering to the Z data (rows)
z_all = z_all[unique_indices, :]

if not np.isfinite(z_all).all():
    print(f"Warning: Found {np.isnan(z_all).sum()} NaNs and {np.isinf(z_all).sum()} Infs in Z-data.")
    print("-> Replacing with zeros/finite values to prevent crash.")
    z_all = np.nan_to_num(z_all, nan=0.0, posinf=np.nanmax(z_all[np.isfinite(z_all)]), neginf=0.0)

print(f"Final Cleaned Shape: {y_centers.shape}")

Found turning point at index 1756. Slicing last sweep.
Sweep is High->Low. Reversing to fit plot axes.
Final Y shape: (153,)
-> Replacing with zeros/finite values to prevent crash.
Final Cleaned Shape: (152,)


In [32]:
# --- SWEEP SELECTION & CLEANING BLOCK ---

# 1. Define Parameters
# Change this index to select which sweep you want to analyze!
#  0  = First sweep
# -1  = The very last (partial) sweep
# -2  = The last COMPLETE sweep (usually what you want)
SWEEP_INDEX = -2 

# 2. Find Turning Points (Peaks and Valleys)
# We use prominence to ignore tiny noise jitters, only finding real turns.
# Adjust prominence if your field range is very small (e.g. 0.001)
prom = (np.max(y_centers) - np.min(y_centers)) * 0.05 
pks_max, _ = find_peaks(y_centers, prominence=prom)     # Tops
pks_min, _ = find_peaks(-y_centers, prominence=prom)    # Bottoms

# Combine all cut-points: [Start, Valley1, Peak1, Valley2, ..., End]
cut_indices = np.concatenate(([0], pks_max, pks_min, [len(y_centers)-1]))
cut_indices = np.sort(np.unique(cut_indices))

# 3. Analyze Available Sweeps
print(f"{'ID':<4} | {'Start Idx':<10} | {'End Idx':<10} | {'Range (T)':<20} | {'Direction'}")
print("-" * 65)

valid_sweeps = []
for i in range(len(cut_indices) - 1):
    start, end = cut_indices[i], cut_indices[i+1]
    # Skip tiny segments (noise or glitches at ends)
    if end - start < 10: continue
    
    val_start = y_centers[start]
    val_end = y_centers[end]
    direction = "UP" if val_end > val_start else "DOWN"
    
    print(f"{i:<4} | {start:<10} | {end:<10} | {val_start:.3f} -> {val_end:.3f}   | {direction}")
    valid_sweeps.append((start, end, direction))

# 4. Select the Sweep
try:
    # Handle negative indexing (e.g. -1 for last)
    if SWEEP_INDEX < 0:
        actual_idx = len(valid_sweeps) + SWEEP_INDEX
    else:
        actual_idx = SWEEP_INDEX
        
    s_start, s_end, s_dir = valid_sweeps[actual_idx]
    
    print("-" * 65)
    print(f"SELECTED SWEEP [{actual_idx}]: {y_centers[s_start]:.3f} -> {y_centers[s_end]:.3f} ({s_dir})")

    # 5. Slice and Orient Data
    # We add +1 to end index to include the last point
    y_centers = y_centers[s_start : s_end+1]
    z_all = z_all[s_start : s_end+1, :]
    
    # If it's a DOWN sweep (High -> Low), flip it so it becomes Monotonic Increasing
    # This ensures pcolormesh and all logic works without modification.
    if s_dir == "DOWN":
        print("-> Flipping DOWN sweep to increasing order...")
        y_centers = y_centers[::-1]
        z_all = z_all[::-1, :]

    # Final Cleanup (remove accidental duplicates from the slice)
    y_centers, unique_idx = np.unique(y_centers, return_index=True)
    z_all = z_all[unique_idx, :]
    
    print(f"Final Data Shape: {y_centers.shape}")

except IndexError:
    print(f"\nERROR: SWEEP_INDEX {SWEEP_INDEX} is out of range. Found {len(valid_sweeps)} sweeps.")
    # Fallback to full data if selection fails (prevents crash)
    pass

ID   | Start Idx  | End Idx    | Range (T)            | Direction
-----------------------------------------------------------------
0    | 0          | 250        | 0.000 -> 1.000   | UP
1    | 250        | 501        | 1.000 -> 0.000   | DOWN
2    | 501        | 752        | 0.000 -> 1.000   | UP
3    | 752        | 1003       | 1.000 -> 0.000   | DOWN
4    | 1003       | 1254       | 0.000 -> 1.000   | UP
5    | 1254       | 1505       | 1.000 -> 0.000   | DOWN
6    | 1505       | 1756       | 0.000 -> 1.000   | UP
7    | 1756       | 1908       | 1.000 -> 0.396   | DOWN
-----------------------------------------------------------------
SELECTED SWEEP [6]: 0.000 -> 1.000 (UP)
Final Data Shape: (251,)


In [ ]:
from matplotlib.legend_handler import HandlerTuple
from matplotlib.transforms import Affine2D

class HandlerVerticalTuple(HandlerTuple):
    def __init__(self, vertical_pad=0.5, y_offset=-2.0, **kwargs):
        self.vertical_pad = vertical_pad
        self.y_offset = y_offset # Manual adjustment for vertical centering
        super().__init__(**kwargs)

    def create_artists(self, legend, orig_handle,
                       xdescent, ydescent, width, height, fontsize, trans):
        n_elements = len(orig_handle)
        artists = []
        
        for i, handle in enumerate(orig_handle):
            # Get the handler for the individual line/item
            handler = legend.get_legend_handler(legend.get_legend_handler_map(), handle)
            a_list = handler.create_artists(legend, handle, xdescent, ydescent, 
                                            width, height, fontsize, trans)
            
            # 1. Calculate the spread (the gap between the two lines)
            spread_offset = ((n_elements - 1) / 2.0 - i) * (fontsize * self.vertical_pad)
            
            # 2. Combine spread with the global y_offset to center against text
            # We add ydescent to stay relative to the handlebox's actual vertical center
            total_v_shift = spread_offset + self.y_offset
            
            for a in a_list:
                # Apply the combined translation
                a.set_transform(Affine2D().translate(0, total_v_shift) + trans)
                artists.append(a)
                
        return artists

# Interactive g-factor

In [33]:
d_x_min, d_x_max = np.min(x_centers), np.max(x_centers)
d_y_min, d_y_max = np.min(y_centers), np.max(y_centers)

mu_B_uev = 57.88
initial_alpha = 0.01

# --- 1. FIGURE ---
fig = plt.figure(figsize=(10, 7))
plt.subplots_adjust(bottom=0.35, top=0.92, right=0.75, left=0.1)
ax = fig.add_axes([0.10, 0.35, 0.52, 0.60])

# Plot
pcm = ax.pcolormesh(x_centers, y_centers, z_all, cmap='binary', shading='auto')
cbar = fig.colorbar(pcm, cax=fig.add_axes([0.63, 0.35, 0.015, 0.60]), label="dI/dV")

# Artists
# 1. Fitted Lines
lines_art = [ax.plot([], [], lw=2, alpha=0.9, label=f'Line {i+1}')[0] for i in range(4)]
# 2. Points Used for Fit (Inliers)
pts_art = [ax.plot([], [], 'o', ms=4, alpha=0.9, markeredgecolor='k', markeredgewidth=0.5, label='Inliers')[0] for _ in range(4)]
# 3. Rejected Points (Faint Red)
pts_rejected = ax.plot([], [], '+', c='red', ms=3, alpha=0.3, label='Rejected')[0]
# 4. Unfitted Points (Strong signal but no line match - Gray x)
pts_unfitted = ax.plot([], [], 'x', c='gray', ms=4, alpha=0.6, label='Unfitted')[0]

ax.set_xlabel("$V_{PG}$ (V)", fontsize=12)
ax.set_ylabel("$B$ (T)", fontsize=12)

# Text
txt_info = fig.text(0.76, 0.70, "Click 'FIT LINES'", family='monospace', fontsize=10, va='top')
txt_res = fig.text(0.76, 0.62, "", fontsize=11, va='top', bbox=dict(facecolor='wheat', alpha=0.3))

# --- 2. CONTROLS ---
# Checkbox for Deviation Mode
ax_chk_dev = fig.add_axes([0.74, 0.28, 0.18, 0.04], frameon=False)
chk_dev = CheckButtons(ax_chk_dev, ['Use Deviation'], [False]) # Default to True (Dev Mode)

def update_zmin_slider(label):
    is_dev_mode = chk_dev.get_status()[0]
    
    if is_dev_mode:
        median_val = np.median(z_all)
        max_dev = np.max(np.abs(z_all - median_val))
        
        new_min = 0.0
        new_max = max_dev
        new_label = 'Min Dev'
    else:
        new_min = np.min(z_all)
        new_max = np.max(z_all)
        new_label = 'Min Z'

    sl_zmin.valmin = new_min
    sl_zmin.valmax = new_max
    
    sl_zmin.ax.set_xlim(new_min, new_max)
    
    sl_zmin.label.set_text(new_label)

    if sl_zmin.val < new_min:
        sl_zmin.set_val(new_min)
    elif sl_zmin.val > new_max:
        sl_zmin.set_val(new_max)
    else:
        sl_zmin.ax.figure.canvas.draw_idle()

chk_dev.on_clicked(update_zmin_slider)

sliders = []
def add_sl(label, vmin, vmax, vinit, y, step=None):
    ax_s = fig.add_axes([0.74, y, 0.18, 0.02])
    s = Slider(ax_s, label, vmin, vmax, valinit=vinit, valstep=step)
    sliders.append(s)
    return s

# View Sliders
sl_xmin = add_sl('X Min', d_x_min, d_x_max, d_x_min, 0.88)
sl_xmax = add_sl('X Max', d_x_min, d_x_max, d_x_max, 0.85)
sl_ymin = add_sl('Y Min', d_y_min, d_y_max, d_y_min, 0.82)
sl_ymax = add_sl('Y Max', d_y_min, d_y_max, d_y_max, 0.79)

# Param Sliders
sl_zmin = add_sl('Min Z', np.min(z_all), np.max(z_all), 0.2*(np.max(z_all) - np.min(z_all)), 0.25)
sl_d = add_sl('Min Dist', 1, 50, 1, 0.22, step=1)
sl_stride = add_sl('Y-Stride', 1, 30, 2, 0.19, step=1) 
sl_percentile = add_sl('Keep Top %', 0, 100, 100, 0.16, step=0.1) 

text_alpha = TextBox(fig.add_axes([0.76, 0.08, 0.10, 0.04]), r"$\alpha$:", initial=str(initial_alpha))
btn_fit = Button(fig.add_axes([0.76, 0.02, 0.18, 0.05]), 'FIT LINES', color='lightblue', hovercolor='0.9')

# --- 3. TOGGLE VISIBILITY ---
ax_check = fig.add_axes([0.76, 0.35, 0.15, 0.10], frameon=False)
check = CheckButtons(ax_check, ['Show Lines', 'Show Used Pts', 'Show Rejected', 'Show Unfitted'], [True, True, True, True])

def toggle_vis(label):
    if label == 'Show Lines':
        for l in lines_art: l.set_visible(not l.get_visible())
    elif label == 'Show Used Pts':
        for p in pts_art: p.set_visible(not p.get_visible())
    elif label == 'Show Rejected':
        pts_rejected.set_visible(not pts_rejected.get_visible())
    elif label == 'Show Unfitted':
        pts_unfitted.set_visible(not pts_unfitted.get_visible())
    fig.canvas.draw_idle()

check.on_clicked(toggle_vis)

# --- 4. UPDATE FUNCTIONS ---
def update_view(val):
    # 1. Get View Limits
    x1, x2 = sl_xmin.val, sl_xmax.val
    y1, y2 = sl_ymin.val, sl_ymax.val

    # 2. Update Plot Axes
    ax.set_xlim(x1, x2)
    ax.set_ylim(y1, y2)

    # 3. Dynamic Color Scaling
    # Create boolean masks to identify data inside the current view
    mask_x = (x_centers >= x1) & (x_centers <= x2)
    mask_y = (y_centers >= y1) & (y_centers <= y2)

    # Check if we have any data in view to avoid errors
    if np.any(mask_x) and np.any(mask_y):
        # Slice the Z-data using np.ix_ (intersections of rows/cols)
        z_view = z_all[np.ix_(mask_y, mask_x)]

        if z_view.size > 0:
            # Find min/max within this slice
            vmin_new = np.min(z_view)
            vmax_new = np.max(z_view)
            
            # Apply to the image artist
            pcm.set_clim(vmin_new, vmax_new)

    fig.canvas.draw_idle()

for s in sliders[:4]: s.on_changed(update_view)

def get_indices(axis, vmin, vmax):
    # Robustly find closest indices regardless of sorting
    idx1 = np.abs(axis - vmin).argmin()
    idx2 = np.abs(axis - vmax).argmin()
    return min(idx1, idx2), max(idx1, idx2)

def run_fit(event):
    txt_info.set_text("Scanning...")
    fig.canvas.draw()
    
    # 1. Coordinate Handling
    # Use global indices to prevent slicing errors
    c_start, c_end = get_indices(x_centers, sl_xmin.val, sl_xmax.val)
    r_start, r_end = get_indices(y_centers, sl_ymin.val, sl_ymax.val)
    
    stride = int(sl_stride.val)
    # Ensure we step positively
    if r_start > r_end: r_start, r_end = r_end, r_start
    rows_scan = np.arange(r_start, r_end, stride)
    
    # Collect Points
    all_candidates = [] 
    h_thresh = sl_zmin.val
    dist = int(sl_d.val)
    
    for r in rows_scan:
        trace_slice = z_all[r, c_start:c_end]
        
        if len(trace_slice) < 3: continue

        use_deviation = chk_dev.get_status()[0]

        if use_deviation:
            # 1. Determine background (median is robust against outliers/peaks)
            bg_level = np.median(trace_slice)
            
            # 2. Create deviation trace
            trace_slice = np.abs(trace_slice - bg_level)

        pks, props = find_peaks(trace_slice, height=h_thresh, distance=dist)
        
        if len(pks) > 0:
            heights = props['peak_heights']
            for pk_rel, h in zip(pks, heights):
                # pk_rel is the index *inside the slice*
                # global_pk is the index in x_centers
                global_pk = c_start + pk_rel
                
                # Check bounds
                if global_pk < len(x_centers):
                    real_x = x_centers[global_pk]
                    real_y = y_centers[r]
                    all_candidates.append([real_x, real_y, h])

    all_candidates = np.array(all_candidates)
    
    if len(all_candidates) < 10:
        txt_info.set_text(f"Found {len(all_candidates)} pts.")
        return

    # 2. Global Filter
    perc = 100 - sl_percentile.val
    z_thresh = np.percentile(all_candidates[:, 2], perc)
    
    mask_strong = all_candidates[:, 2] >= z_thresh
    best_pts = all_candidates[mask_strong]
    rejected = all_candidates[~mask_strong]
    
    pts_rejected.set_data(rejected[:,0], rejected[:,1])
    
    # 3. RANSAC Fit Setup
    data_for_fit = best_pts[:, :2]
    remaining = data_for_fit.copy()
    
    # Clear old
    for art in lines_art + pts_art: art.set_data([], [])
    
    ransac = RANSACRegressor(min_samples=5, residual_threshold=0.002, random_state=42)
    
    # Store results temporarily
    found_lines = []
    
    # 4. Loop to Find lines (just collect them first)
    for i in range(4):
        if len(remaining) < 6: break
        try:
            X = remaining[:, 1].reshape(-1, 1) # B field
            y = remaining[:, 0]                # V_pg
            
            ransac.fit(X, y)
            mask = ransac.inlier_mask_
            if np.sum(mask) < 6: break

            m_prime = ransac.estimator_.coef_[0]
            c_prime = ransac.estimator_.intercept_
            
            # Save the inliers and model for later
            inliers = remaining[mask]
            found_lines.append({
                'm': m_prime,
                'c': c_prime,
                'inliers': inliers
            })
            
            remaining = remaining[~mask]
        except:
            break

    # 5. Plot "Unfitted" Points (Strong points left over)
    if len(remaining) > 0:
        pts_unfitted.set_data(remaining[:,0], remaining[:,1])

    # 5. Sort, Group, and Plot
    if len(found_lines) > 0:
        # Sort by Voltage Intercept (Low V -> High V)
        found_lines.sort(key=lambda x: x['c'])
        
        # Define Color Groups
        # Group 1 (Low V): Cool colors
        # Group 2 (High V): Warm colors
        colors = ['cyan', 'blue', 'orange', 'red'] 
        
        g_vals = []
        
        try: alpha_val = float(text_alpha.text)
        except: alpha_val = initial_alpha

        for i, line in enumerate(found_lines):
            if i >= 4: break # Safety limit
            
            # Physics
            g = (2 * alpha_val * 1e6 * abs(line['m'])) / mu_B_uev
            g_vals.append(g)
            
            # Assign Color
            # If there are 4 lines: 0,1 are Group 1 (Blue). 2,3 are Group 2 (Red).
            c = colors[i]
            
            # Plot Inliers
            pts_art[i].set_data(line['inliers'][:,0], line['inliers'][:,1])
            pts_art[i].set_color(c)
            pts_art[i].set_alpha(0.8)
            
            # Plot Fitted Line
            y_plot = np.linspace(sl_ymin.val, sl_ymax.val, 100)
            x_plot = line['m'] * y_plot + line['c']
            lines_art[i].set_data(x_plot, y_plot)
            lines_art[i].set_color(c)
            lines_art[i].set_linewidth(2.5)

        # 6. Math Display
        res_str = r"$\alpha \Delta V_{PG} = \frac{1}{2} g \mu_B \Delta B$" + "\n"
        
        if len(g_vals) == 4:
            g1 = np.mean(g_vals[:2]) # First 2 (Cyan/Blue)
            g2 = np.mean(g_vals[2:]) # Last 2 (Orange/Red)
            
            g_v = (g1 + g2) / 2
            g_s = (g1 - g2) / 2
            
            res_str += (
                r"$g_1 \approx " + f"{g1:.3f}" + r" \quad g_2 \approx " + f"{g2:.3f}" + r"$" + "\n"
                r"$g_v = \frac{g_1 + g_2}{2} = \mathbf{" + f"{g_v:.3f}" + "}$" + "\n"
                r"$g_s = \frac{g_1 - g_2}{2} = \mathbf{" + f"{g_s:.3f}" + "}$"
            )
        else:
            avg_g = np.mean(g_vals)
            res_str += r"$\bar{g} \approx " + f"{avg_g:.3f}$" + f"\n({len(g_vals)} lines)"

        txt_res.set_text(res_str)
        txt_res.set_bbox(dict(facecolor='lightgreen', alpha=0.5, edgecolor='green'))
        txt_res.set_fontsize(11)

        # 7. Update Legend with Explicit Groups
        from matplotlib.legend_handler import HandlerTuple

        handles = [
            (lines_art[0], lines_art[1]), 
            (lines_art[2], lines_art[3]), 
            pts_rejected, 
            pts_unfitted
        ]

        labels = [
            r'$g_1$ Group (Low V)', 
            r'$g_2$ Group (High V)', 
            'Rejected Pts', 
            'Unfitted Pts'
        ]

        ax.legend(
            handles=handles, 
            labels=labels, 
            loc='upper left', 
            fontsize=10,
            handler_map={tuple: HandlerVerticalTuple(vertical_pad=0.4, y_offset=-4)}, 
            handleheight=3  # Increase this to give the stack room to breathe
        )

    else:
        txt_res.set_text("No lines fitted.")
        txt_res.set_bbox(dict(facecolor='wheat', alpha=0.3))
        
    txt_info.set_text(f"Scanned {len(all_candidates)} pts.\nKept {len(best_pts)} (Top {100-perc:.0f}%).")
    
    toggle_vis(None)
    fig.canvas.draw_idle()

btn_fit.on_clicked(run_fit)

update_view(None)
plt.show(block=True)

# Output